In [ ]:
pip install pyDOE

In [ ]:
import jax
import jax.numpy as jnp
from flax import linen as nn
import optax
from typing import Callable

from functools import partial

import tensorflow as tf
from pyDOE import lhs
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from google. colab import runtime

In [ ]:
# Boundaries of the computational domain
xleft, xright = -np.pi, np.pi
t0, tfinal = 0, 1.0

In [ ]:
# Wave speed
c = 1

# Maximum number of waves in ICs
max_waves = 3

In [ ]:
# How many sensor points to use
N_sensor_points = 100

In [ ]:
def u0(x, amplitudes, phases):
  u = 0.0*x
  for i in range(max_waves):
      u += amplitudes[i]*jnp.sin((i+1)*x + phases[i])
  # if x.ndim == 1:
  #   for i in range(max_waves):
  #     u += amplitudes[i]*jnp.sin((i+1)*x + phases[i])
  # else:
  #   for i in range(max_waves):
  #     u += amplitudes[:,i:i+1]*jnp.sin((i+1)*x + phases[:,i:i+1])
  return u

def u0v(x, amplitudes, phases):
  u = 0.0*x
  for i in range(max_waves):
    u += amplitudes[:,i:i+1]*jnp.sin((i+1)*x + phases[:,i:i+1])
  return u

In [ ]:
# Define the normalization layer
class Normalize(nn.Module):
  xmin: float
  xmax: float

  @nn.compact
  def __call__(self, x):
    return 2.0*(x-self.xmin)/(self.xmax - self.xmin) - 1.0

In [ ]:
class HardConstraint(nn.Module):
  t0: float
  tfinal: float
  u0: Callable

  @nn.compact
  def __call__(self, inputs):
    t, x, nn, amps, phs = inputs
    return self.u0(x, amps, phs) + t/self.tfinal*nn

In [ ]:
# Define the periodic boundary constraint layer
class PeriodicBCs(nn.Module):
  @nn.compact
  def __call__(self, x):
    return jnp.concatenate((jnp.cos(x), jnp.sin(x)), axis=1)

In [ ]:
# Define the Fourier layer

class FourierLayer(nn.Module):

  k: int
  dz: int

  @nn.compact
  def __call__(self, inp):

    b = inp
    jfft = jnp.fft.fft(b, axis = 0)
    jfft = jfft.at[:, self.k:].set(0)
    jiift = jnp.fft.ifft(jfft, axis = 0)


    out = jiift
    return out



In [ ]:
class FourierNetwork(nn.Module):

  layers: int
  k: int
  dz: int

  @nn.compact
  def __call__(self, inp):
    b = inp
    for i in range(self.layers):
      b = FourierLayer(self.k, self.dz)(b)
    b = jnp.real(b)
    out = b
    return out

In [ ]:
class CombineBranches(nn.Module):

  @nn.compact
  def __call__(self, inp1, inp2):
    mult = jnp.sum(inp1*inp2)
    # For consistency, we add the batch dimension back here
    out = jnp.reshape(mult, (-1,1))
    return out

In [ ]:
class z_layer(nn.Module) :

  dz: int

  @nn.compact
  def __call__(self, inp):
    b = inp
    num_inp = b.shape[0]
    b = jnp.reshape(b, (num_inp, 1))
    b = nn.Dense(self.dz)(b)
    b = nn.relu(b)
    out = b
    return out


In [ ]:
class q_layer(nn.Module) :

  layers: int
  units: int

  @nn.compact
  def __call__(self, inp):
    b = inp
    for i in range(self.layers-1):
      b = nn.Dense(self.units)(b)
      b = nn.tanh(b)
    out = nn.Dense(self.units)(b)
    out = jnp.reshape(out, (-1,))
    return out

In [ ]:
class MLP(nn.Module):

  layers: int
  units: int

  @nn.compact
  def __call__(self, inp):
    b = inp
    for i in range(self.layers-1):
      b = nn.Dense(self.units)(b)
      b = nn.tanh(b)
    out = nn.Dense(self.units)(b)
    return out

In [ ]:
class MLPBranch(nn.Module):

  layers: int
  units: int

  @nn.compact
  def __call__(self, inp):
    b = inp
    num_in = b.shape[0]
    b = jnp.reshape(b, (num_in, 1))
    for i in range(self.layers-1):
      b = nn.Dense(self.units)(b)
      b = nn.tanh(b)
    out = nn.Dense(self.units)(b)
    return out

In [ ]:
# class DeepONet(nn.Module):

#   t0: float
#   tfinal: float
#   layers: int
#   units: int

#   @nn.compact
#   def __call__(self, t, x, u):

#     # Add the batch dimension
#     t = jnp.reshape(t, (-1,1))
#     x = jnp.reshape(x, (-1,1))

#     b1 = Normalize(self.t0, self.tfinal)(t)
#     b2 = PeriodicBCs()(x)

#     b = jnp.concatenate((b1, b2), axis=1)

#     trunk_net = MLP(self.layers, self.units)(b)
#     branch_net = MLP(self.layers, self.units)(u)

#     out = CombineBranches()(trunk_net, branch_net)
#     # out = HardConstraint(self.t0, self.tfinal, self.u0)([t, x, out, amps, phs])

#     # Remove the batch dimension
#     out = jnp.reshape(out, (-1,))

#     return out

In [ ]:
class DeepONet(nn.Module):

  t0: float
  tfinal: float
  layers: int
  units: int
  k: int
  dz: int

  @nn.compact
  def __call__(self, t, x, u):

    # Add the batch dimension
    t = jnp.reshape(t, (-1,1))
    x = jnp.reshape(x, (-1,1))

    print("t shape", t.shape)
    print("x shape", x.shape)
    print("u shape", u.shape)

    b1 = Normalize(self.t0, self.tfinal)(t)
    b2 = PeriodicBCs()(x)


    b = jnp.concatenate((b1, b2), axis=1)

    trunk_net = MLP(self.layers, self.units)(b)
    # trunk_net = jnp.repeat(trunk_net, 3, axis = 0)

    print("Trunknet shape", trunk_net.shape)

    Zlayer = z_layer(self.dz)(u)
    print("zlayer shape", Zlayer.shape)
    fourier_network = FourierNetwork(6, self.k, self.dz)(Zlayer)
    print("fourier_net shape", fourier_network.shape)
    Qlayer = q_layer(self.layers, 1)(fourier_network)
    print("Qlayer shape", Qlayer.shape)
    branch_net = MLPBranch(1, self.units)(Qlayer)
    print("branch_net output", branch_net.shape)

    out = CombineBranches()(trunk_net, branch_net)
    # out = HardConstraint(self.t0, self.tfinal, self.u0)([t, x, out, amps, phs])

    # Remove the batch dimension
    out = jnp.reshape(out, (-1,))
    print("output shape", out.shape)

    return out

In [ ]:

# Create the model
deeponet = DeepONet(t0, tfinal, layers=6, units=40, k = 20, dz = 40)

# Initialize the model parameters
params = deeponet.init(jax.random.PRNGKey(0), 1.0, 1.0, jnp.ones((100,)))

In [ ]:
print(deeponet.tabulate(jax.random.PRNGKey(0), jnp.ones((1,)), jnp.ones((1,)), jnp.ones((100,))))

In [ ]:
def defineCollocationPoints(N=10000, max_waves=5):

  # Uniform random sampling for PDE points
  tx_min = np.array([t0, xleft])
  tx_max = np.array([tfinal, xright])
  pde_points = tx_min + (tx_max - tx_min)*lhs(2, N)
  t_pde = pde_points[:, 0]
  x_pde = pde_points[:, 1]

  # Initial data sample
  x_points = np.linspace(xleft, xright, N_sensor_points)

  amplitudes = np.random.randn(N, max_waves)
  phases = -np.pi*np.random.rand(N, max_waves) + jnp.pi/2

  # Generate the data for the branch net
  usensors = np.float32(u0v(np.repeat(np.expand_dims(x_points, axis=0), repeats=N, axis=0), amplitudes, phases))
  pdes = np.column_stack([t_pde, x_pde, amplitudes, phases]).astype(np.float32)

  return (pdes, usensors)

In [ ]:
N = 20000
(pdes, usensors) = defineCollocationPoints(N, max_waves)

In [ ]:
# Helper functions
@jax.jit
def u_model(t, x, u, amps, phases, params):

  return u0(x, amps, phases) + t/tfinal*deeponet.apply(params, t, x, u)[0]

@jax.jit
def ut(t, x, u, amps, phases, params):
  return jax.vmap(jax.grad(u_model, 0), [0,0,0,0,0,None])(t.reshape(-1), x.reshape(-1), u,
                                                          amps, phases, params).reshape(x.shape)
@jax.jit
def ux(t, x, u, amps, phases, params):
  return jax.vmap(jax.grad(u_model, 1), [0,0,0,0,0,None])(t.reshape(-1), x.reshape(-1), u,
                                                          amps, phases, params).reshape(x.shape)

@jax.jit
def loss(params, t, x, u, amps, phases):
  return jnp.mean(jnp.square(ut(t, x, u, amps, phases, params) + ux(t, x, u, amps, phases, params)))

@jax.jit
def train_step(params, pdes, u):
  t, x = pdes[:,0], pdes[:,1]
  amps, phases = pdes[:,2:2+max_waves], pdes[:,2+max_waves:]
  return jax.value_and_grad(loss)(params, t, x, u, amps, phases)

@partial(jax.jit, static_argnums=(3,))
def optimize(grads, opt_state, params, optimizer_update):
  updates, opt_state = optimizer_update(grads, opt_state, params)
  params = optax.apply_updates(params, updates)
  return params, opt_state

In [ ]:
def train(params, pde_points, u_sensors, epochs=10000):

  # Number of ICs and total number of PDE collocation points
  N = pde_points.shape[0]

  # Batch size for PDE points
  batch_size = 1000
  nr_batches = N//batch_size

  ds_u = tf.data.Dataset.from_tensor_slices(u_sensors)
  ds_pde = tf.data.Dataset.from_tensor_slices(pde_points)

  print(ds_u)
  print(ds_pde)
  input()

  ds = tf.data.Dataset.zip((ds_pde, ds_u))
  print(ds)
  ds = ds.cache().shuffle(N).batch(batch_size)
  ds = ds.prefetch(tf.data.experimental.AUTOTUNE)

  # Optimizer and learning rate to be used
  lr = 1e-3

  optimizer = optax.adam(lr)
  opt_state = optimizer.init(params)

  epoch_loss = np.zeros(epochs)

  # Main training loop
  for i in range(epochs):

    print(i)

    for (pdes_batch, u_sensor_batch) in ds:

      loss, grads = train_step(params, pdes_batch.numpy(), u_sensor_batch.numpy())
      params, opt_state = optimize(grads, opt_state, params, optimizer.update)

      epoch_loss[i] += loss

    epoch_loss[i] /= nr_batches

    print(epoch_loss[i])
    if i % 100 == 0:
      print(f'Loss in epoch {i}: {epoch_loss[i]}')

  return params, epoch_loss

In [ ]:
epochs = 100
params, loss = train(params, pdes, usensors, epochs)

Loss in epoch 0: 5.716349411010742
Loss in epoch 100: 0.020035287737846373
Loss in epoch 200: 0.00925212949514389
Loss in epoch 300: 0.009843189269304276
Loss in epoch 400: 0.0143770232796669
Loss in epoch 500: 0.008809623122215272
Loss in epoch 600: 0.0071566089987754825
Loss in epoch 700: 0.00975092276930809
Loss in epoch 800: 0.014087285101413726
Loss in epoch 900: 0.01343664973974228
Loss in epoch 1000: 0.010362344235181809
Loss in epoch 1100: 0.007797317951917649
Loss in epoch 1200: 0.013267554342746735
Loss in epoch 1300: 0.00509972870349884
Loss in epoch 1400: 0.008524317294359207
Loss in epoch 1500: 0.008363907039165498
Loss in epoch 1600: 0.00929926112294197
Loss in epoch 1700: 0.011150572448968887
Loss in epoch 1800: 0.011154066026210784
Loss in epoch 1900: 0.006536766886711121
Loss in epoch 2000: 0.006725212931632996
Loss in epoch 2100: 0.009676218777894974
Loss in epoch 2200: 0.00791403353214264
Loss in epoch 2300: 0.0062876008450984955
Loss in epoch 2400: 0.01202797219157219
Loss in epoch 2500: 0.008334121853113174
Loss in epoch 2600: 0.0048028726130723955
Loss in epoch 2700: 0.005183598771691322
Loss in epoch 2800: 0.0067517593502998356
Loss in epoch 2900: 0.0044997245073318485
Loss in epoch 3000: 0.006330671906471253
Loss in epoch 3100: 0.005344024673104286
Loss in epoch 3200: 0.0050753321498632434
Loss in epoch 3300: 0.005287832766771317
Loss in epoch 3400: 0.006473276764154434
Loss in epoch 3500: 0.004470688477158547
Loss in epoch 3600: 0.00669245719909668
Loss in epoch 3700: 0.004563459381461144
Loss in epoch 3800: 0.010114722698926926
Loss in epoch 3900: 0.00856357514858246
Loss in epoch 4000: 0.008476946502923965
Loss in epoch 4100: 0.006182970851659775
Loss in epoch 4200: 0.006488572061061859
Loss in epoch 4300: 0.0078626848757267
Loss in epoch 4400: 0.008105431497097016
Loss in epoch 4500: 0.0045359663665294645
Loss in epoch 4600: 0.007601768523454666
Loss in epoch 4700: 0.006086596846580505
Loss in epoch 4800: 0.00621282123029232
Loss in epoch 4900: 0.004038559272885323



In [ ]:
plt.semilogy(np.arange(0, epochs, 1), loss)
plt.grid()
plt.xlabel('epochs')
plt.ylabel('loss')

In [ ]:
# Sample a new IC
amplitudes = np.random.randn(max_waves, 1)
phases = -np.pi*np.random.rand(max_waves, 1) + np.pi/2

x_points = np.linspace(xleft, xright, N_sensor_points)


# Predict with trained PINN
l, n = 50, N_sensor_points
t = np.linspace(t0, tfinal, l)
x = np.linspace(xleft, xright, n)
T, X  = np.meshgrid(t, x, indexing='ij')

tt = T.flatten()
xx = X.flatten()

usensor = np.repeat(np.expand_dims(u0(x_points, amplitudes, phases), axis=0), len(tt), axis=0)
amps = np.repeat(np.expand_dims(amplitudes[:,0], axis=0), len(tt), axis=0)
phs = np.repeat(np.expand_dims(phases[:,0], axis=0), len(tt), axis=0)

# u = jax.vmap(u_model, [0, 0, 0, 0, 0, None])(tt, xx, usensor, amps, phs, params)
# u = np.reshape(u, (l, n))

print(amplitudes.shape, phases.shape, X.shape, T.shape)

uTrue = u0(X-c*T, amplitudes, phases)

In [ ]:
print(amplitudes.shape, phases.shape, X.shape, T.shape)

In [ ]:
uTrue = u0(X-c*T, amplitudes, phases)

In [ ]:
plt.figure(figsize=(15,5))
plt.subplot(131)

plt.contourf(T, X, u, 20, cmap=plt.cm.coolwarm)
plt.xlabel('t')
plt.ylabel('x')
plt.title('NN solution')
plt.colorbar()


plt.subplot(132)
plt.contourf(T, X, uTrue, 20, cmap=plt.cm.coolwarm)
plt.xlabel('t')
plt.ylabel('x')
plt.title('True solution')
plt.colorbar()

plt.subplot(133)
plt.contourf(T, X, u-uTrue, 20, cmap=plt.cm.coolwarm)
plt.xlabel('t')
plt.ylabel('x')
plt.title(f'MAE: {tf.reduce_mean(tf.abs(u-uTrue)): 2.4f}')
plt.colorbar()

In [ ]:
plt.plot(x, u[-1,],'k')
plt.plot(x, uTrue[-1,],'k--')
plt.grid()
plt.legend(['NN','GT'])

### Time stepping with DeepOnet

In [ ]:
steps = 10

In [ ]:
u = []
for i in range(steps):
  u_c = jax.vmap(u_model, [0, 0, 0, 0, 0, None])(tt, xx, usensor, amps, phs, params)
  u_c = np.reshape(u_c, (l, n))

  # Reconstruct solution
  uinit = np.repeat(usensor[-1:,], repeats=l, axis=0)

  # For form u(t,x) = u0(x) + t/tf*v(t,x)
  u_c = uinit - u0(X, amplitudes, phases) + u_c
  u.append(u_c[:-1,])

  usensor = np.repeat(u_c[-1:,:], len(tt), axis=0)

u = np.concatenate(u)

In [ ]:
# Exact solution
l, n = 50, N_sensor_points
t = np.linspace(t0, steps*tfinal, steps*(l-1))
x = np.linspace(xleft, xright, n)
TAll, XAll  = np.meshgrid(t, x, indexing='ij')

uTrue = u0(XAll-c*TAll, amplitudes, phases)

In [ ]:
plt.figure(figsize=(15,5))
plt.subplot(131)

plt.contourf(TAll, XAll, u, 20, cmap=plt.cm.coolwarm)
plt.xlabel('t')
plt.ylabel('x')
plt.title('NN solution')
plt.colorbar()


plt.subplot(132)
plt.contourf(TAll, XAll, uTrue, 20, cmap=plt.cm.coolwarm)
plt.xlabel('t')
plt.ylabel('x')
plt.title('True solution')
plt.colorbar()

plt.subplot(133)
plt.contourf(TAll, XAll, u-uTrue, 20, cmap=plt.cm.coolwarm)
plt.xlabel('t')
plt.ylabel('x')
plt.title(f'MAE: {tf.reduce_mean(tf.abs(u-uTrue)): 2.4f}')
plt.colorbar()

In [ ]:
plt.plot(x, u[-1,],'k')
plt.plot(x, uTrue[-1,],'k--')
plt.grid()
plt.legend(['NN','GT'])

In [ ]:
layer = nn.Dense(features=4)

params = layer.init(jax.random.key(0), jnp.ones((3,)))

jax.tree_map(jnp.shape, params)